In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Shape: {df.shape}")
print("Loaded ✅")

Shape: (7043, 21)
Loaded ✅


In [2]:
df = df.drop(columns=['customerID'])
print("customerID dropped ✅")

customerID dropped ✅


In [4]:
# It's stored as string — convert to number
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many became NaN
print(f"NaN in TotalCharges: {df['TotalCharges'].isnull().sum()}")

# Fill with median (the modern, clean way without inplace=True)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print("TotalCharges fixed ✅")

NaN in TotalCharges: 11
TotalCharges fixed ✅


In [8]:
print(df.dtypes)
# Updated to explicitly include both 'object' and 'str' to satisfy modern Pandas rules
print(f"\nObject columns: {list(df.select_dtypes(include=['object', 'str']).columns)}")

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

Object columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [9]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    df[col] = (df[col] == 'Yes').astype(int)

print("Binary columns encoded ✅")
print(df[binary_cols].head(3))

Binary columns encoded ✅
   Partner  Dependents  PhoneService  PaperlessBilling
0        1           0             0                 1
1        0           0             1                 0
2        0           0             1                 1


In [10]:
df['gender'] = (df['gender'] == 'Male').astype(int)
print("Gender encoded: Male=1, Female=0 ✅")

Gender encoded: Male=1, Female=0 ✅


In [11]:
# These have 3+ unique values — need one-hot encoding
multi_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'
]

df = pd.get_dummies(df, columns=multi_cols, drop_first=False)

print(f"New shape after encoding: {df.shape}")
print("Categorical columns encoded ✅")

New shape after encoding: (7043, 41)
Categorical columns encoded ✅


In [12]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Numeric features scaled ✅")
print(df[scale_cols].describe().round(2))

Numeric features scaled ✅
        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7043.00
mean     -0.00           -0.00         -0.00
std       1.00            1.00          1.00
min      -1.32           -1.55         -1.00
25%      -0.95           -0.97         -0.83
50%      -0.14            0.19         -0.39
75%       0.92            0.83          0.66
max       1.61            1.79          2.83


In [13]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nFeature list:\n{list(X.columns)}")

Features (X): (7043, 40)
Target (y): (7043,)

Feature list:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check'

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keeps churn ratio same in both splits
)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size:  {X_test.shape[0]} rows")
print(f"\nChurn rate in train: {y_train.mean()*100:.1f}%")
print(f"Churn rate in test:  {y_test.mean()*100:.1f}%")

Train size: 5634 rows
Test size:  1409 rows

Churn rate in train: 26.5%
Churn rate in test:  26.5%


In [15]:
import joblib

# Save splits for use in Phase 3
joblib.dump((X_train, X_test, y_train, y_test), 'processed_data.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Data saved to processed_data.pkl ✅")
print("Scaler saved to scaler.pkl ✅")
print("\n→ Ready for Phase 3: Modeling 🚀")

Data saved to processed_data.pkl ✅
Scaler saved to scaler.pkl ✅

→ Ready for Phase 3: Modeling 🚀
